In [1]:
import duckdb
import pandas as pd
import requests
import json
from datetime import datetime, timedelta
import os
from pathlib import Path
import geopandas as gpd
import xml.etree.ElementTree as ET
import re

In [2]:
import requests
import pandas as pd


# 1. Configuración
# ID de ejemplo para Renta Media por hogar por Distritos (Verifica el ID actual en la web del INE)
# A fecha de hoy, un ID común para renta por distritos suele rondar los 30xxx o 37xxx

TABLE_ID = "30656"  # 30824 total albacete : 30656
BASE_URL = f"https://servicios.ine.es/wstempus/js/es/DATOS_TABLA/{TABLE_ID}?tip=AM"


# 2. Hacemos la petición
try:
    # request json file of the table 
    response = requests.get(BASE_URL)
    response.raise_for_status()
    resultados = []
    # print(response.text)
    # print(type(response.json()))
    for entrada in response.json():
        # Filtro: Solo Distritos
        # print(entrada)
        es_distrito = False
        codigo_distrito = "N/A"
        nombre_distrito = "N/A"
        
        #Test metadata is not empty
        metadata = entrada.get("MetaData", [])
        if metadata is None: continue
        for meta in entrada.get("MetaData", []):
            if meta.get("T3_Variable") == "Distritos":
                es_distrito = True
                codigo_distrito = meta.get("Codigo")
                nombre_distrito = meta.get("Nombre")
                break
        
        if not es_distrito:
            continue

        # Limpieza del nombre del concepto
        nombre_completo = entrada.get("Nombre", "")
        try:
            concepto = nombre_completo.split('.')[-2].strip()
        except:
            concepto = nombre_completo
        if concepto != "Renta neta media por persona":
            continue
        print(concepto)
        # Filtro de Años
        for dato in entrada.get("Data", []):
            anyo = dato.get("Anyo")
            if 2022 <= anyo <= 2025:
                resultados.append({
                    "ine_district": codigo_distrito,
                    "name": nombre_distrito,
                    "concept": concepto,
                    "year": anyo,
                    "avg_net_income": dato.get("Valor")
                })


    df = pd.DataFrame(resultados)

    df_pivot = df.pivot_table(
                index=['ine_district', 'name', 'concept'],
                columns='year',
                values='avg_net_income'
            ).reset_index()
except Exception as e:
    print(f"Error: {e}")

Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta media por persona
Renta neta med

In [4]:
df_pivot 

NameError: name 'df_pivot' is not defined

In [25]:
df_pivot.head(50)

year,ine_id_district,name,concept,2022,2023
0,0200101,Abengibre distrito 01,Renta neta media por persona,13063.0,14105.0
1,0200201,Alatoz distrito 01,Renta neta media por persona,11346.0,12222.0
2,0200301,Albacete distrito 01,Renta neta media por persona,15595.0,16364.0
3,0200302,Albacete distrito 02,Renta neta media por persona,13076.0,13772.0
4,0200303,Albacete distrito 03,Renta neta media por persona,12572.0,13194.0
5,0200304,Albacete distrito 04,Renta neta media por persona,12382.0,13062.0
6,0200305,Albacete distrito 05,Renta neta media por persona,12706.0,13424.0
7,0200306,Albacete distrito 06,Renta neta media por persona,18505.0,19545.0
8,0200307,Albacete distrito 07,Renta neta media por persona,13188.0,14092.0
9,0200308,Albacete distrito 08,Renta neta media por persona,12548.0,13485.0


In [1]:
import requests
import pandas as pd
from pyjstat import pyjstat

# TABLA DE POBLACION

TABLE_ID = "69095"  # 65031 total, albacete : 69095
URL = f"https://servicios.ine.es/wstempus/js/es/DATOS_TABLA/{TABLE_ID}?tip=AM"


# 2. Hacemos la petición
try:
    # request json file of the table 
    response = requests.get(URL)
    response.raise_for_status()
    resultados = []
    # print(response.text)
    # print(type(response.json()))
    for entrada in response.json():
        metadata = entrada.get("MetaData", [])
        
        # --- 1. FILTRO: Solo Secciones ---
        es_seccion = False
        codigo_seccion = ""
        nombre_seccion = ""
        
        for meta in metadata:
            if meta.get("T3_Variable") == "Secciones":
                es_seccion = True
                codigo_seccion = meta.get("Codigo")
                nombre_seccion = meta.get("Nombre")
                break
        
        if not es_seccion:
            continue # Saltamos si es Municipio, Provincia, etc.

        # --- 2. FILTRO: Solo 'Total' (Sin distinción de género) ---
        es_total_sexo = False
        for meta in metadata:
            if meta.get("T3_Variable") == "Sexo":
                if meta.get("Nombre") == "Total":
                    es_total_sexo = True
                break
        
        if not es_total_sexo:
            continue # Saltamos si es "Hombres" o "Mujeres"

        # --- 3. DEFINIR CONCEPTO (¿Es población Total, Española o Extranjera?) ---
        # Buscamos la variable "Países" o "Nacionalidad" para limpiar el nombre
        es_total_nacion = False
        concepto = "Poblacion Total" # Valor por defecto
        for meta in metadata:
            # print(meta.get("T3_Variable"))
            if meta.get("T3_Variable") in ["Países","Nacionalidad"]:
                if meta.get("Nombre") == "Total":
                    es_total_nacion = True

        if not es_total_nacion :
            continue
        # --- 4. EXTRACCIÓN DE DATOS ---
        for dato in entrada.get("Data", []):
            anyo = dato.get("Anyo")
            if anyo: # Asegurar que existe el año
                resultados.append({
                    "ine_section": codigo_seccion,
                    "name": nombre_seccion,
                    "concept": concepto,
                    "year": anyo,
                    "total_population": dato.get("Valor")
                })

    # --- CREACIÓN DE TABLA PIVOTADA ---
    if resultados:
        df = pd.DataFrame(resultados)
        
        # Pivotamos para poner los años en columnas
        df_pivot = df.pivot_table(
            index=['ine_section', 'name', 'concept'], 
            columns='year', 
            values='total_population'
        ).reset_index()
        
        # Quitamos el nombre del eje de columnas para que quede limpio
        df_pivot.columns.name = None
        

    else:
        print("No se encontraron datos que cumplan los criterios.")
except Exception as e:
    print(f"Error: {e}")

In [2]:
df_pivot

,ine_section,name,concept,2021,2022,2023,2024,2025
0,0200101001,Abengibre sección 01001,Poblacion Total,752.0,739.0,768.0,760.0,747.0
1,0200201001,Alatoz sección 01001,Poblacion Total,510.0,502.0,503.0,505.0,506.0
2,0200301001,Albacete sección 01001,Poblacion Total,848.0,838.0,830.0,848.0,835.0
3,0200301002,Albacete sección 01002,Poblacion Total,1264.0,1219.0,1202.0,1202.0,1177.0
4,0200301003,Albacete sección 01003,Poblacion Total,2077.0,2048.0,2056.0,1997.0,2004.0
...,...,...,...,...,...,...,...,...
292,0208501001,Viveros sección 01001,Poblacion Total,332.0,335.0,308.0,305.0,294.0
293,0208601001,Yeste sección 01001,Poblacion Total,1542.0,1505.0,1489.0,1515.0,1468.0
294,0208601003,Yeste sección 01003,Poblacion Total,1042.0,1010.0,1015.0,1015.0,1014.0
295,0290101001,Pozo Cañada sección 01001,Poblacion Total,1511.0,1473.0,1428.0,1416.0,1409.0


In [29]:
import duckdb
import pandas as pd
import requests
import json
from datetime import datetime, timedelta
import os
from pathlib import Path
import geopandas as gpd
import xml.etree.ElementTree as ET
import re
from pathlib import Path


In [30]:
con = duckdb.connect()
con.sql("INSTALL ducklake; LOAD ducklake;")
con.sql("INSTALL spatial; LOAD spatial;")

In [31]:
# This for Detaching from ducklake in case
con.sql(f"""
USE memory;
DETACH my_ducklake;
    """)

BinderException: Binder Error: Failed to detach database with name "my_ducklake": database not found

In [32]:
# Ataching to local duck lake
con.sql(f"""
ATTACH 'ducklake:my_ducklake.ducklake' AS my_ducklake;

USE my_ducklake;
    """)

In [40]:
con.sql(f"""
    SELECT * FROM df_pivot
    """)

┌─────────────┬────────────────────────────────┬─────────────────┬────────┬────────┬────────┬────────┐
│ ine_section │              name              │     concept     │  2021  │  2022  │  2023  │  2024  │
│   varchar   │            varchar             │     varchar     │ double │ double │ double │ double │
├─────────────┼────────────────────────────────┼─────────────────┼────────┼────────┼────────┼────────┤
│ 0100101001  │ Alegría-Dulantzi sección 01001 │ Poblacion-Total │ 1378.0 │ 1389.0 │ 1390.0 │ 1398.0 │
│ 0100101002  │ Alegría-Dulantzi sección 01002 │ Poblacion-Total │ 1546.0 │ 1580.0 │ 1579.0 │ 1567.0 │
│ 0100201001  │ Amurrio sección 01001          │ Poblacion-Total │ 1880.0 │ 1873.0 │ 1869.0 │ 1871.0 │
│ 0100201002  │ Amurrio sección 01002          │ Poblacion-Total │ 1568.0 │ 1561.0 │ 1554.0 │ 1539.0 │
│ 0100201003  │ Amurrio sección 01003          │ Poblacion-Total │ 1506.0 │ 1503.0 │ 1524.0 │ 1531.0 │
│ 0100201004  │ Amurrio sección 01004          │ Poblacion-Total │ 1811.0

In [35]:
table_name = "ine_population"
con.sql(f"""
            CREATE TABLE IF NOT EXISTS bronze.{table_name} AS
            (SELECT * FROM df_pivot) LIMIT 0
            """)

In [36]:
con.sql("DESCRIBE bronze.ine_population")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ ine_section │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ name        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ concept     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ 2021        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ 2022        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ 2023        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ 2024        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘